# B2S 08 - AndinaLog HR Drivers

Conversión Bronze a Silver de atributos laborales por conductor. La fuente se lee sin modificarse, los nombres se eliminan por minimización de PII y toda transformación o règle de enrutamiento queda auditada sin valores personales.

In [9]:
import os
from pathlib import Path
from datetime import datetime, timezone
import json
import re
import pandas as pd

def detectar_raiz():
    candidatas = []
    if os.getenv('ANDINALOG_ROOT'):
        candidatas.append(Path(os.environ['ANDINALOG_ROOT']))
    cwd = Path.cwd().resolve()
    candidatas.extend([cwd, *cwd.parents])
    contenido = Path('/content').resolve()
    candidatas.extend([contenido, *contenido.parents])
    if '__file__' in globals():
        archivo = Path(__file__).resolve()
        candidatas.extend([archivo.parent, *archivo.parents])
    relativa = Path('datos/bronze/andinalog_hr_drivers.csv')
    visitadas = set()
    for candidata in candidatas:
        normalizada = candidata.resolve()
        if normalizada in visitadas:
            continue
        visitadas.add(normalizada)
        if (normalizada / relativa).exists():
            return normalizada
    raise FileNotFoundError('No se encontró datos/bronze/andinalog_hr_drivers.csv')

CONFIG = {
    'bronze': 'datos/bronze/andinalog_hr_drivers.csv',
    'notebook': 'notebooks/bronze_silver/08_hr_drivers/B2S_08_HR_Drivers.ipynb',
    'silver': 'datos/silver/andinalog_HR_Drivers_silver.csv',
    'quarantine': 'datos/quarantine/andinalog_HR_Drivers_quarantine.csv',
    'informe': 'informes/bronze_silver/Informe_B2S_08_HR_Drivers.md',
    'columnas_requeridas': ['chofer_id', 'chofer_nombre', 'centro_distribucion', 'horas_conduccion_mes', 'salario_base_bob', 'ausentismo_dias'],
    'columnas_salida': ['chofer_id', 'centro_distribucion', 'horas_conduccion_mes', 'salario_base_bob', 'ausentismo_dias'],
    'centros_conocidos': ['Cochabamba', 'La Paz', 'Santa Cruz', 'Oruro', 'Tarija'],
    'patron_chofer_id': r'^CHO-\d{3}$',
    'centinelas': [-999],
    'imputaciones': {},
    'criterio_duplicado': 'Copias exactas de atributos operativos: conservar la primera y enviar las copias adicionales a cuarentena; grupos conflictivos se bloquean completos.',
    'fuentes_auxiliares': {
        'Informe_B2S_06_Warehouse_Costs.md': 'contextual, noaru',
        'B2S_06_Andinalog_Warehouse_Costs.ipynb': 'contextual, noaru',
        'andinalog_warehouse_costs_quarantine.csv': 'contextual, noaru',
        'andinalog_warehouse_costs_silver.csv': 'contextual, noaru'
    },
}
CONFIG['fuentes_auxiliares'] = {clave: 'contextual, no utilizado' for clave in CONFIG['fuentes_auxiliares']}
RAIZ = detectar_raiz()
for clave in ['silver', 'quarantine', 'informe']:
    (RAIZ / CONFIG[clave]).parent.mkdir(parents=True, exist_ok=True)
print(f'Raíz: {RAIZ}')
print('Perfil: andinalog_hr_drivers.csv | Granularidad: un conductor por chofer_id')

Raíz: C:\Users\remrodri\Github\practicasNotebookColab\proyecto-integradorV2
Perfil: andinalog_hr_drivers.csv | Granularidad: un conductor por chofer_id


In [10]:
bronze = pd.read_csv(RAIZ / CONFIG['bronze'], dtype=str, keep_default_na=False)
faltantes = [columna for columna in CONFIG['columnas_requeridas'] if columna not in bronze.columns]
if faltantes:
    raise ValueError(f'Columnas requeridas ausentes: {faltantes}')
bronze.insert(0, '_fila_bronze', range(1, len(bronze) + 1))
perfil_seguro = {
    'filas': len(bronze),
    'columnas': len(bronze.columns),
    'nombres_presentes': int(bronze['chofer_nombre'].str.strip().ne('').sum()),
    'ids_unicos_raw': bool(bronze['chofer_id'].is_unique),
    'ids_unicos_normalizados': bool(bronze['chofer_id'].str.strip().str.upper().is_unique),
    'centros_observados': sorted(bronze['centro_distribucion'].unique().tolist()),
}
print(json.dumps(perfil_seguro, ensure_ascii=False, indent=2))
bronze[['chofer_nombre']].astype(str).apply(lambda columna: columna.str.len().max()).rename('longitud_maxima_nombre').to_frame()

{
  "filas": 156,
  "columnas": 7,
  "nombres_presentes": 156,
  "ids_unicos_raw": false,
  "ids_unicos_normalizados": false,
  "centros_observados": [
    "Cochabamba",
    "La Paz",
    "Oruro",
    "Santa Cruz",
    "Tarija"
  ]
}


,longitud_maxima_nombre
chofer_nombre,12


In [11]:
def estructurar(df):
    out = df.copy()
    out['_fila_fuente'] = out['_fila_bronze'] + 1
    out['chofer_id_original'] = out['chofer_id']
    out['centro_distribucion_original'] = out['centro_distribucion']
    for columna in ['horas_conduccion_mes', 'salario_base_bob', 'ausentismo_dias']:
        out[f'{columna}_original'] = out[columna]
    return out.drop(columns=['chofer_nombre'])

def normalizar(df):
    out = df.copy()
    out['chofer_id'] = out['chofer_id_original'].str.strip().str.upper()
    out['centro_distribucion'] = out['centro_distribucion_original'].str.strip()
    out['chofer_id_normalizado'] = out['chofer_id'].ne(out['chofer_id_original'])
    return out

def convertir_numericas(df):
    out = df.copy()
    centinelas = set(CONFIG['centinelas'])
    for columna in ['horas_conduccion_mes', 'salario_base_bob', 'ausentismo_dias']:
        original = out[f'{columna}_original']
        convertido = pd.to_numeric(original.replace('', pd.NA), errors='coerce')
        centinela = convertido.isin(centinelas)
        out[columna] = convertido.mask(centinela)
        out[f'{columna}_conversion_invalida'] = convertido.isna() & original.str.strip().ne('') & ~centinela
        out[f'{columna}_centinela_detectado'] = centinela
    return out

def acumular_motivos(serie):
    return serie.map(lambda motivos: ';'.join(motivos) if motivos else '')

def validar_y_clasificar(df):
    out = df.copy()
    error_id = out['chofer_id'].eq('') | ~out['chofer_id'].str.fullmatch(CONFIG['patron_chofer_id'], na=False)
    error_centro = ~out['centro_distribucion'].isin(CONFIG['centros_conocidos'])
    errores = [[] for _ in range(len(out))]
    motivos = [[] for _ in range(len(out))]
    for posicion, valor in enumerate(error_id):
        if valor:
            errores[posicion].append('chofer_id:formato_invalido')
    for posicion, valor in enumerate(error_centro):
        if valor:
            errores[posicion].append('centro_distribucion:referencia_invalida')
    for columna, regla in [('horas_conduccion_mes', lambda x: pd.isna(x) or x < 0), ('salario_base_bob', lambda x: pd.isna(x) or x <= 0), ('ausentismo_dias', lambda x: pd.isna(x) or x < 0 or float(x).is_integer() is False)]:
        invalida = out[f'{columna}_conversion_invalida'] | out[columna].map(regla)
        for posicion, valor in enumerate(invalida):
            if valor:
                errores[posicion].append(f'{columna}:conversion_invalida_o_rango_invalido')
    columnas_operativas = CONFIG['columnas_salida']
    out['duplicado_operativo'] = out.duplicated(subset=columnas_operativas, keep=False)
    out['copia_duplicada'] = out.duplicated(subset=columnas_operativas, keep='first')
    perfiles = out.groupby('chofer_id', dropna=False)[columnas_operativas[1:]].apply(lambda grupo: len(grupo.drop_duplicates()))
    out['conflicto_clave'] = out['chofer_id'].map(perfiles).gt(1)
    for posicion in out.index[out['copia_duplicada']]:
        errores[posicion].append('chofer_id:duplicado_exacto_copia_resuelta')
    for posicion in out.index[out['conflicto_clave']]:
        errores[posicion].append('chofer_id:duplicado_conflictivo')
    for posicion in range(len(out)):
        motivos[posicion].append('nombre_eliminado_por_privacidad')
        if bool(out.iloc[posicion]['chofer_id_normalizado']):
            motivos[posicion].append('chofer_id:espacios_y_mayusculas_normalizados')
    out['errores_bloqueantes'] = pd.Series([ ';'.join(valores) for valores in errores], index=out.index)
    out['motivos_transformacion'] = pd.Series([ ';'.join(valores) for valores in motivos], index=out.index)
    out['motivos_imputacion'] = ''
    out['fue_transformada'] = out['chofer_id_normalizado']
    out['fue_imputada'] = False
    out['imputacion_metodo'] = ''
    out['imputacion_motivo'] = ''
    out['calidad_motivo'] = out['errores_bloqueantes'].replace('', 'sin_incidencias')
    out['calidad_estado'] = out['errores_bloqueantes'].eq('').map({True: 'valida', False: 'cuarentena'})
    out['conteo_transformaciones'] = out['motivos_transformacion'].ne('').astype(int) + out['chofer_id_normalizado'].astype(int)
    out['conteo_imputaciones'] = 0
    return out

work = bronze.pipe(estructurar).pipe(normalizar).pipe(convertir_numericas).pipe(validar_y_clasificar)
silver = work.loc[work['errores_bloqueantes'].eq('')].copy()
quarantine = work.loc[work['errores_bloqueantes'].ne('')].copy()
print(silver['calidad_estado'].value_counts().to_string())
print(quarantine['calidad_motivo'].value_counts().to_string())

calidad_estado
valida    151
calidad_motivo
chofer_id:duplicado_exacto_copia_resuelta    5


In [12]:
columna_tecnica = ['_fila_fuente']
columnas_auditoria = [
    'chofer_id_original', 'centro_distribucion_original', 'horas_conduccion_mes_original', 'salario_base_bob_original', 'ausentismo_dias_original',
    'errores_bloqueantes', 'motivos_transformacion', 'motivos_imputacion', 'fue_transformada', 'fue_imputada',
    'chofer_id_normalizado', 'horas_conduccion_mes_conversion_invalida', 'horas_conduccion_mes_centinela_detectado',
    'salario_base_bob_conversion_invalida', 'salario_base_bob_centinela_detectado', 'ausentismo_dias_conversion_invalida', 'ausentismo_dias_centinela_detectado',
    'duplicado_operativo', 'copia_duplicada', 'conflicto_clave', 'calidad_motivo', 'calidad_estado',
    'conteo_transformaciones', 'conteo_imputaciones', 'imputacion_metodo', 'imputacion_motivo'
]
columnas_salida = columna_tecnica + CONFIG['columnas_salida'] + columnas_auditoria
silver = silver[columnas_salida].copy()
quarantine = quarantine[columnas_salida].copy()
silver.to_csv(RAIZ / CONFIG['silver'], index=False, encoding='utf-8')
quarantine.to_csv(RAIZ / CONFIG['quarantine'], index=False, encoding='utf-8')
conciliacion = pd.DataFrame([{'bronze': len(bronze), 'silver': len(silver), 'cuarentena': len(quarantine)}])
print(conciliacion.to_string(index=False))
print(f'Duplicados operativos observados: {int(work["duplicado_operativo"].sum())} | copias en cuarentena: {int(work["copia_duplicada"].sum())}')

 bronze  silver  cuarentena
    156     151           5
Duplicados operativos observados: 10 | copias en cuarentena: 5


In [13]:
fecha_utc = datetime.now(timezone.utc).isoformat()
motivos_cuarentena = quarantine['errores_bloqueantes'].value_counts().to_dict()
reporte = f'''# Informe B2S 08 - AndinaLog HR Drivers

## Objetivo, entidad y granularidad
Conversión auditada de atributos laborales desde Bronze CSV hacia Silver y cuarentena.
- Entidad: conductor con un identificador operativo no personal.
- Granularidad: una fila por `chofer_id` después de resolver copias exactas.
- Clave funcional: `chofer_id`.

## Perfil Bronze de esta ejecución
- Filas Bronze: {len(bronze)}.
- Columnas Bronze: {len(CONFIG['columnas_requeridas'])}.
- Centros observados: {', '.join(sorted(bronze['centro_distribucion'].unique()))}.
- Identificadores normalizados: {int(work['chofer_id_normalizado'].sum())}.
- Filas involucradas en duplicados operativos: {int(work['duplicado_operativo'].sum())}.
- Conflictos de clave: {int(work['conflicto_clave'].sum())}.

## Reglas, privacidad y transformaciones
- Bronze se lee como texto y no se modifica.
- `chofer_nombre` se elimina antes de generar Silver o cuarentena por minimización de PII.
- `chofer_id` se normaliza mediante recorte y mayúsculas; no se imputan identificadores.
- Los centros se validan contra el catálogo configurado.
- Horas, salario y ausentismo conservan original, valor tratado y banderas de conversión y centinela.
- Las copias exactas conservan la primera instancia en Silver y envían copias adicionales a cuarentena.

## Imputaciones
- No se realizaron imputaciones.
- No se usaron cero, medias, medianas, modas, `ffill`, `bfill` ni información futura.

## Evidencia de contradicción con el plan previo
- El plan indicó 157 filas; el archivo real contiene {len(bronze)}. La conciliación de esta ejecución prevalece.
- El plan atribuyó cinco casos a identificadores inválidos; la inspección encontró cinco grupos de copias operativas exactas.
- Se normalizó un identificador con espacios y minúsculas; no fue enviado a cuarentena por ese motivo.

## Integridad referencial y uso de auxiliares
- Los cinco grupos duplicados presentan campos operativos idénticos y no constituyen conflictos.
- Warehouse Costs 06 se considera contextual y no se utiliza: no aporta claves ni atributos necesarios para HR Drivers.
- No se usan variables laborales para sancionar conductores ni se afirma causalidad.

## Resultado y conciliación
- Silver: {len(silver)} filas.
- Cuarentena: {len(quarantine)} filas.
- Conciliación: Bronze {len(bronze)} = Silver {len(silver)} + cuarentena {len(quarantine)}.
- Motivos de cuarentena: {motivos_cuarentena}.

## Limitaciones
- La fuente es contextual y no debe entrar a Gold sin necesidad analítica demostrada y proporcional.
- Los atributos laborales no sustituyen evidencia de cumplimiento, безопасidad o causalidad.

## Archivos generados
- `{CONFIG['notebook']}`
- `{CONFIG['silver']}`
- `{CONFIG['quarantine']}`
- `{CONFIG['informe']}`

## Reproducibilidad
- Fecha UTC: {fecha_utc}.
- Python: {__import__('sys').version.split()[0]}.
- pandas: {pd.__version__}.
- Ejecutar las celdas en orden desde la raíz del proyecto o desde el directorio del notebook; `detectar_raiz()` busca la raíz entre los directorios actuales y sus ancestros.
- Los controles finales se aplican sobre los CSV persistidos.
'''
(RAIZ / CONFIG['informe']).write_text(reporte, encoding='utf-8')
print(f'Informe generado: {CONFIG["informe"]}')

Informe generado: informes/bronze_silver/Informe_B2S_08_HR_Drivers.md


In [14]:
silver_persistido = pd.read_csv(RAIZ / CONFIG['silver'], dtype=str, keep_default_na=False)
quarantine_persistido = pd.read_csv(RAIZ / CONFIG['quarantine'], dtype=str, keep_default_na=False)
assert len(bronze) == len(silver_persistido) + len(quarantine_persistido)
assert silver_persistido['chofer_id'].is_unique
assert silver_persistido['errores_bloqueantes'].eq('').all()
assert silver_persistido['calidad_estado'].eq('valida').all()
assert quarantine_persistido['calidad_estado'].eq('cuarentena').all()
assert not any('chofer_nombre' in columna for columna in silver_persistido.columns)
assert not any('chofer_nombre' in columna for columna in quarantine_persistido.columns)
assert set(silver_persistido['_fila_fuente']).isdisjoint(set(quarantine_persistido['_fila_fuente']))
assert set(silver_persistido['_fila_fuente']) | set(quarantine_persistido['_fila_fuente']) == set(str(x) for x in range(2, len(bronze) + 2))
assert (RAIZ / CONFIG['informe']).exists()
print('CONTROLES_OK')
print(pd.DataFrame([{'bronze': len(bronze), 'silver': len(silver_persistido), 'cuarentena': len(quarantine_persistido), 'errores_bloqueantes_silver': int(silver_persistido['errores_bloqueantes'].ne('').sum())}]).to_string(index=False))

CONTROLES_OK
 bronze  silver  cuarentena  errores_bloqueantes_silver
    156     151           5                           0
